# ⚠️ RUN CELLS IN ORDER: 1 → 2 → 3 → etc.

This notebook compares two local SQLite databases and consolidates them into one.

**Important**: Execute cells sequentially from top to bottom!

In [3]:
# STEP 1: Import Required Libraries
import sqlite3
import pandas as pd
import os
import shutil

print("✅ Libraries imported successfully")
print("\n📊 Comparing two local SQLite databases:")
print("  1. /househelp.db (root)")
print("  2. /instance/househelp.db (instance)")


✅ Libraries imported successfully

📊 Comparing two local SQLite databases:
  1. /househelp.db (root)
  2. /instance/househelp.db (instance)


In [4]:
# STEP 2: Connect to Both SQLite Databases
sqlite_root = "househelp.db"
sqlite_instance = "instance/househelp.db"

conn_root = None
conn_instance = None

if os.path.exists(sqlite_root):
    conn_root = sqlite3.connect(sqlite_root)
    print(f"✅ Connected to Root SQLite: {sqlite_root}")
else:
    print(f"❌ Root SQLite not found: {sqlite_root}")

if os.path.exists(sqlite_instance):
    conn_instance = sqlite3.connect(sqlite_instance)
    print(f"✅ Connected to Instance SQLite: {sqlite_instance}")
else:
    print(f"❌ Instance SQLite not found: {sqlite_instance}")


✅ Connected to Root SQLite: househelp.db
✅ Connected to Instance SQLite: instance/househelp.db


In [5]:
# STEP 3: Query SQLite - Tables & Comparison
print("\n" + "="*60)
print("📋 TABLE STRUCTURES COMPARISON")
print("="*60)

for db_name, conn in [("ROOT", conn_root), ("INSTANCE", conn_instance)]:
    if conn:
        cursor = conn.cursor()
        cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
        tables = cursor.fetchall()
        print(f"\n{db_name} Database Tables:")
        for table in tables:
            print(f"  - {table[0]}")



📋 TABLE STRUCTURES COMPARISON

ROOT Database Tables:
  - helper
  - attendance
  - payment

INSTANCE Database Tables:
  - helper
  - attendance
  - payment


In [6]:
# STEP 4: Query Helper Table - Both Databases
print("\n" + "="*60)
print("👥 HELPER TABLE DATA")
print("="*60)

for db_name, conn in [("ROOT", conn_root), ("INSTANCE", conn_instance)]:
    if conn:
        try:
            query = "SELECT * FROM helper LIMIT 5;"
            df = pd.read_sql_query(query, conn)
            count = pd.read_sql_query("SELECT COUNT(*) as count FROM helper", conn)['count'][0]
            
            print(f"\n{db_name} Database - Helper Table:")
            print(df.to_string())
            print(f"Total rows: {count}")
        except Exception as e:
            print(f"{db_name}: Error - {e}")



👥 HELPER TABLE DATA

ROOT Database - Helper Table:
   id   name         type  default_qty
0   1  Pintu  milk vendor          1.0
Total rows: 1

INSTANCE Database - Helper Table:
   id    name             type  default_qty
0   1   Pintu      milk vendor          1.0
1   2   Seema  domestic helper          1.0
2   3  Saumya        Caretaker          1.0
Total rows: 3


In [7]:
# STEP 5: Summary & Data Sync Status
print("\n" + "="*60)
print("🔄 DATA SYNCHRONIZATION CHECK")
print("="*60)

def count_table(conn, table_name):
    try:
        cursor = conn.cursor()
        cursor.execute(f"SELECT COUNT(*) FROM {table_name};")
        return cursor.fetchone()[0]
    except:
        return 0

for table in ['helper', 'attendance', 'payment']:
    root_count = count_table(conn_root, table) if conn_root else 0
    instance_count = count_table(conn_instance, table) if conn_instance else 0
    
    sync_status = "✅ SYNCED" if root_count == instance_count else "⚠️  DIFFERENT"
    print(f"\n{table.upper()}:")
    print(f"  Root count: {root_count}")
    print(f"  Instance count: {instance_count}")
    print(f"  Status: {sync_status}")



🔄 DATA SYNCHRONIZATION CHECK

HELPER:
  Root count: 1
  Instance count: 3
  Status: ⚠️  DIFFERENT

ATTENDANCE:
  Root count: 0
  Instance count: 18
  Status: ⚠️  DIFFERENT

PAYMENT:
  Root count: 0
  Instance count: 1
  Status: ⚠️  DIFFERENT


## Database Consolidation Decision

The `/instance/househelp.db` appears to be a legacy or backup database. We'll consolidate and keep only the root database.

**Next Steps**: Run cells below to decide which DB to keep and consolidate safely.


In [8]:
# STEP 6: Consolidation Decision - Which Database to Keep?
print("\n" + "="*60)
print("🔍 CONSOLIDATION RECOMMENDATION")
print("="*60)

def get_latest_date(conn, table, date_col):
    try:
        cursor = conn.cursor()
        cursor.execute(f"SELECT MAX({date_col}) FROM {table};")
        return cursor.fetchone()[0]
    except:
        return None

if conn_root and conn_instance:
    root_latest_att = get_latest_date(conn_root, 'attendance', 'date')
    instance_latest_att = get_latest_date(conn_instance, 'attendance', 'date')
    
    print(f"\nRoot DB - Latest attendance date: {root_latest_att}")
    print(f"Instance DB - Latest attendance date: {instance_latest_att}")
    
    if root_latest_att and instance_latest_att:
        if root_latest_att >= instance_latest_att:
            print("\n✅ RECOMMENDATION: KEEP Root database (has newer or equal data)")
            print("❌ REMOVE: Instance database (older or redundant)")
        else:
            print("\n✅ RECOMMENDATION: KEEP Instance database (has newer data)")
            print("❌ REMOVE: Root database (older or redundant)")
    else:
        print("\n⚠️ One or both databases may be empty")



🔍 CONSOLIDATION RECOMMENDATION

Root DB - Latest attendance date: None
Instance DB - Latest attendance date: 2026-05-05

⚠️ One or both databases may be empty


In [9]:
# STEP 7: Safe Consolidation - Backup and Remove Duplicate DB
print("\n" + "="*60)
print("🔧 CONSOLIDATION ACTION")
print("="*60)

backup_path = "instance/househelp.db.backup"
instance_path = "instance/househelp.db"

# Close connections before file operations
if conn_root:
    conn_root.close()
    print("✅ Closed Root DB connection")
if conn_instance:
    conn_instance.close()
    print("✅ Closed Instance DB connection")

print("\n⚠️  STEP 1: Creating backup of instance database...")
if os.path.exists(instance_path):
    try:
        shutil.copy2(instance_path, backup_path)
        print(f"✅ Backup created: {backup_path}")
    except Exception as e:
        print(f"❌ Backup failed: {e}")
else:
    print(f"Instance database not found: {instance_path}")

print("\n⚠️  STEP 2: Deleting instance database (keeping root only)...")
if os.path.exists(instance_path):
    try:
        os.remove(instance_path)
        print(f"✅ Deleted: {instance_path}")
        print("\n🎉 DATABASE CONSOLIDATED!")
        print("   Now using only: /househelp.db (root)")
    except Exception as e:
        print(f"❌ Deletion failed: {e}\nManually delete if desired: {instance_path}")
else:
    print("✅ Instance database already removed or doesn't exist")



🔧 CONSOLIDATION ACTION
✅ Closed Root DB connection
✅ Closed Instance DB connection

⚠️  STEP 1: Creating backup of instance database...
✅ Backup created: instance/househelp.db.backup

⚠️  STEP 2: Deleting instance database (keeping root only)...
✅ Deleted: instance/househelp.db

🎉 DATABASE CONSOLIDATED!
   Now using only: /househelp.db (root)
